# L6 + bge_ft — Phân tích đặc trưng mới (`bge_ft`)

Trực quan hoá & phân tích **7 đặc trưng mới `bge_ft`** mà phương pháp **L6-final+bge_ft** (§23) thêm vào L6 LightGBM LambdaRank — tín hiệu từ retriever fine-tuned **`sr-emb-bge-v1`**.

**Nguồn dữ liệu:** `results/m4_v2/cache/ltr_features_bgeft/{ds}.npz` (6 dataset, **52 cột** = 45 cũ + 7 `bge_ft`), sinh bởi `src/kmeans/scripts/ltr/add_bgeft_features.py` từ cache 45-cột `ltr_features/` + `results/models/sr-emb-bge-v1` + `results/bge/corpus_ids.json`.

Mỗi dòng = 1 cặp (query, candidate) trong **M4 Stage-1 pool** (~500/query); `y=1` nếu candidate là gold. 7 cột `bge_ft` được **nối vào cuối** nên cache 45-cột cũ và mọi `column_indices()` cũ vẫn hợp lệ (chọn nhóm KHÔNG có `bge_ft` ⇒ tái lập đúng L6 baseline).

**Mục tiêu phân tích:** (1) 7 đặc trưng `bge_ft` trông như thế nào; (2) chúng tách gold tốt ra sao; (3) có phải tín hiệu *mới* (khác `bge_score` zero-shot) hay không; (4) L6 thực sự dùng chúng nhiều thế nào (gain importance); (5) vì sao R@1 nhảy 48→62 và nDCG@10 67→81.

> Ghi chú: notebook chạy trên **H100** nơi có `results/`. Nếu thiếu cache, các ô sẽ in hướng dẫn sinh dữ liệu.

## 0. Setup

In [ ]:
import sys, os, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# tìm repo root (chứa 'src') rồi nạp package kmeans
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); os.chdir(ROOT)

# Sau khi tổ chức lại package: ltr_features nằm ở kmeans.ltr
# (shim `from kmeans import ltr_features` cũng vẫn hoạt động).
from kmeans.ltr import ltr_features

DATASETS = ['theoremqa', 'logicbench', 'toolqa', 'champ', 'medcalcbench', 'bigcodebench']
FEATS  = ltr_features.ALL_FEATURES                 # 52 = 45 base + 7 bge_ft
GROUPS = ltr_features.FEATURE_GROUPS
BGEFT  = GROUPS['bge_ft']                          # 7 đặc trưng mới
BGE_IDX  = [FEATS.index(f) for f in BGEFT]         # vị trí cột (7 cột cuối)
BASE_IDX = [i for i in range(len(FEATS)) if i not in set(BGE_IDX)]   # 45 cột gốc
FEAT2GROUP = {f: g for g, fl in GROUPS.items() for f in fl}

CACHE   = ROOT / 'results/m4_v2/cache/ltr_features_bgeft'   # bảng 52 cột
CACHE45 = ROOT / 'results/m4_v2/cache/ltr_features'         # bảng 45 cột (đối chiếu, optional)
QEMB    = ROOT / 'results/m4_v2/cache/query_emb_ft'         # query embedding fine-tuned

plt.rcParams['figure.dpi'] = 110
print('repo root :', ROOT)
print('features  :', len(FEATS), '| base:', len(BASE_IDX), '| bge_ft:', len(BGE_IDX))
print('bge_ft idx:', BGE_IDX)
print('cache 52  :', 'FOUND' if CACHE.exists() else 'MISSING (sinh trên H100 — xem mục 1)')

### Định nghĩa 7 đặc trưng `bge_ft`

Với mỗi cặp (query *q*, skill *s*) trong pool, `u = qft(q)`, `v = sft(s)` (sr-emb-bge-v1, đã chuẩn hoá L2 ⇒ `u·v` = cosine):

In [ ]:
defs = [
 ('bgeft_cosine',     'u·v — cosine query↔skill (fine-tuned). Tín hiệu CHÍNH.'),
 ('bgeft_rank',       'thứ hạng skill theo cosine TRONG pool của query (1 = cao nhất).'),
 ('bgeft_inv_rank',   '1 / bgeft_rank — nhấn mạnh đỉnh, bão hoà ở đuôi.'),
 ('bgeft_rank_norm',  'bgeft_rank / pool_size — chuẩn hoá [0,1], so sánh được giữa các query.'),
 ('bgeft_is_top1',    '1 nếu skill là argmax cosine của pool, else 0.'),
 ('bgeft_margin_top1','cosine − max_cosine_pool (≤0; =0 cho top-1) — khoảng cách tới đỉnh.'),
 ('bgeft_z',          'z-score cosine trong pool: (cos−mean)/std — độ nổi bật tương đối.'),
]
pd.DataFrame(defs, columns=['feature', 'định nghĩa (within-pool)'])

## 1. Nạp 6 bảng đặc trưng 52 cột

Ghép thành `Xall` (rows×52), `yall` (gold flag), `dsall` (nhãn dataset). `Xb` = riêng khối 7 cột `bge_ft`.

In [ ]:
DATA_OK = CACHE.exists() and all((CACHE / f'{ds}.npz').exists() for ds in DATASETS)
if not DATA_OK:
    print('⚠️  Chưa có cache 52-cột results/m4_v2/cache/ltr_features_bgeft/.')
    print('    Sinh trên H100 (sau git pull, PYTHONPATH=src):')
    print('      python src/kmeans/scripts/ltr/run_fair_eval.py --grid moderate   # -> cache 45-cột ltr_features/')
    print('      python src/kmeans/scripts/ltr/add_bgeft_features.py              # +7 cột -> ltr_features_bgeft/')
    data = {}
else:
    data = {ds: ltr_features.load_features(CACHE / f'{ds}.npz') for ds in DATASETS}
    Xall = np.concatenate([data[ds]['X'] for ds in DATASETS]).astype(np.float32)
    yall = np.concatenate([data[ds]['y'] for ds in DATASETS]).astype(np.int8)
    dsall = np.concatenate([np.full(len(data[ds]['y']), ds) for ds in DATASETS])
    # sanity: 7 cột bge_ft đúng là 7 cột cuối, tên khớp
    assert list(data[DATASETS[0]]['feature_names'][-7:]) == list(BGEFT), 'feature_names mismatch'
    assert Xall.shape[1] == 52, Xall.shape
    Xb = Xall[:, BGE_IDX]   # (rows, 7) — chỉ khối bge_ft, dùng nhiều bên dưới
    print('Xall:', Xall.shape, '| gold:', int(yall.sum()), f'({100*yall.mean():.3f}%)')
    print('khối bge_ft Xb:', Xb.shape)

## 2. Tổng quan theo dataset

Xác nhận **52 cột** ở mọi dataset (45 cũ giữ nguyên + 7 `bge_ft` nối cuối). `pct_gold` ~0.27% (rất mất cân bằng — đúng bản chất reranking).

In [ ]:
if DATA_OK:
    rows = []
    for ds in DATASETS:
        d = data[ds]; n = d['X'].shape[0]; pos = int(d['y'].sum()); q = len(d['group_sizes'])
        rows.append(dict(dataset=ds, rows=n, queries=q, cand_per_query=round(n/q, 1),
                         n_cols=d['X'].shape[1], gold_rows=pos, pct_gold=round(100*pos/n, 3)))
    tot_q = sum(len(data[ds]['group_sizes']) for ds in DATASETS)
    rows.append(dict(dataset='TOTAL', rows=Xall.shape[0], queries=tot_q,
                     cand_per_query=round(Xall.shape[0]/tot_q, 1), n_cols=52,
                     gold_rows=int(yall.sum()), pct_gold=round(100*yall.mean(), 3)))
    display(pd.DataFrame(rows))
else:
    print('(cần dữ liệu — xem mục 1)')

## 3. Phân phối 7 đặc trưng `bge_ft` — gold vs non-gold

Mật độ chuẩn hoá, **đỏ = gold**, xanh = non-gold (subsample 300k dòng, cắt 0.5–99.5 percentile). `bgeft_cosine` và `bgeft_z` của gold lệch hẳn sang phải; `bgeft_rank/rank_norm` của gold dồn về 0 (rank thấp = tốt).

In [ ]:
if DATA_OK:
    rng = np.random.default_rng(0)
    sidx = rng.choice(Xall.shape[0], size=min(300_000, Xall.shape[0]), replace=False)
    Xbs, ys = Xb[sidx], yall[sidx]
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    for i, (name, ax) in enumerate(zip(BGEFT, axes.ravel())):
        col = Xbs[:, i]
        lo, hi = np.percentile(col, [0.5, 99.5])
        bins = np.linspace(lo, hi, 50) if hi > lo else 50
        ax.hist(col[ys == 0], bins=bins, density=True, alpha=0.55, color='steelblue', label='non-gold')
        ax.hist(col[ys == 1], bins=bins, density=True, alpha=0.65, color='crimson', label='gold')
        ax.set_title(f'[{BGE_IDX[i]}] {name}', fontsize=10); ax.tick_params(labelsize=7)
    axes.ravel()[-1].axis('off'); axes.ravel()[0].legend(fontsize=9)
    fig.suptitle('7 đặc trưng bge_ft — phân phối gold (đỏ) vs non-gold (xanh)', y=1.02, fontsize=13)
    plt.tight_layout(); plt.show()
else:
    print('(cần dữ liệu — xem mục 1)')

## 4. `bge_ft` tách gold tốt đến đâu?

(a) **ROC** của `bgeft_cosine` như một bộ phát hiện gold trong pool, so với `bge_score` zero-shot. (b) Histogram **rank của gold** theo `bgeft_rank` (gold có hay nằm top-1?). (c) **Lift** của `bgeft_is_top1`: P(gold | top-1 bge_ft) so với base rate.

In [ ]:
if DATA_OK:
    from sklearn.metrics import roc_auc_score, roc_curve
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    cos = Xall[:, FEATS.index('bgeft_cosine')]
    auc = roc_auc_score(yall, cos); fpr, tpr, _ = roc_curve(yall, cos)
    ax[0].plot(fpr, tpr, color='crimson', lw=2, label=f'bgeft_cosine  AUC={auc:.3f}')
    if 'bge_score' in FEATS:
        c2 = Xall[:, FEATS.index('bge_score')]; a2 = roc_auc_score(yall, c2); f2, t2, _ = roc_curve(yall, c2)
        ax[0].plot(f2, t2, color='steelblue', lw=1.5, ls='--', label=f'bge_score (zero-shot)  AUC={a2:.3f}')
    ax[0].plot([0, 1], [0, 1], 'k:', lw=0.6); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
    ax[0].set_title('ROC — phát hiện gold trong pool'); ax[0].legend()
    grank = Xall[yall == 1, FEATS.index('bgeft_rank')]
    ax[1].hist(np.clip(grank, 1, 100), bins=np.arange(1, 102), color='crimson')
    ax[1].set_title(f'bgeft_rank của gold\nmedian={np.median(grank):.0f} | ≤1: {100*np.mean(grank<=1):.1f}% | ≤10: {100*np.mean(grank<=10):.1f}%')
    ax[1].set_xlabel('rank trong pool (clip@100)'); ax[1].set_ylabel('# gold')
    is_top1 = Xall[:, FEATS.index('bgeft_is_top1')] == 1
    p_top1, base = yall[is_top1].mean(), yall.mean()
    ax[2].bar(['base rate', 'P(gold | bgeft_is_top1)'], [100*base, 100*p_top1], color=['gray', 'crimson'])
    ax[2].set_ylabel('%'); ax[2].set_title(f'Lift của bge_ft top-1: ×{p_top1/base:.0f}')
    for i, v in enumerate([100*base, 100*p_top1]): ax[2].text(i, v, f'{v:.2f}%', ha='center', va='bottom')
    plt.tight_layout(); plt.show()
else:
    print('(cần dữ liệu — xem mục 1)')

## 5. Đặc trưng nào tách gold mạnh nhất? `bge_ft` vs 45 cũ

Standardized mean difference `(mean_gold − mean_nongold)/std` cho cả 52 đặc trưng, tô màu theo nhóm (**đỏ = `bge_ft`**, in đậm). Kỳ vọng cụm `bge_ft` đứng đầu cả hai cực — đó là lý do L6+bge_ft mạnh hơn.

In [ ]:
if DATA_OK:
    from matplotlib.patches import Patch
    mg, mn = Xall[yall == 1].mean(0), Xall[yall == 0].mean(0)
    eff = (mg - mn) / (Xall.std(0) + 1e-9)
    order = np.argsort(eff)
    palette = {'retrieval': '#4C72B0', 'm4': '#55A868', 'a7': '#8172B2',
               'qsc': '#CCB974', 'confidence': '#64B5CD', 'bge_ft': '#C44E52'}
    colors = [palette[FEAT2GROUP[FEATS[i]]] for i in order]
    fig, ax = plt.subplots(figsize=(8.5, 13))
    ax.barh([FEATS[i] for i in order], eff[order], color=colors)
    for lbl, i in zip(ax.get_yticklabels(), order):
        if FEAT2GROUP[FEATS[i]] == 'bge_ft': lbl.set_fontweight('bold')
    ax.axvline(0, color='k', lw=0.6)
    ax.set_title('Standardized mean diff (52 đặc trưng) — đỏ = bge_ft; >0 cao hơn ở gold')
    ax.legend(handles=[Patch(color=c, label=g) for g, c in palette.items()], fontsize=8, loc='lower right')
    ax.tick_params(labelsize=7); plt.tight_layout(); plt.show()
else:
    print('(cần dữ liệu — xem mục 1)')

## 6. `bgeft_cosine` có phải chỉ là `bge_score` cũ? (kiểm tra tín hiệu MỚI)

Tương quan 7 đặc trưng `bge_ft` (hàng) × 45 đặc trưng cũ (cột). Nếu `|corr|` thấp ⇒ `bge_ft` mang thông tin mà 45 đặc trưng cũ KHÔNG có (fine-tune đã học ánh xạ query→gold-skill). Kèm scatter `bgeft_cosine` vs `bge_score`.

In [ ]:
if DATA_OK:
    rng = np.random.default_rng(1)
    s = rng.choice(Xall.shape[0], size=min(150_000, Xall.shape[0]), replace=False)
    C = np.corrcoef(Xall[s].T)                 # 52x52
    sub = C[np.ix_(BGE_IDX, BASE_IDX)]         # 7 x 45
    fig, ax = plt.subplots(figsize=(15, 4))
    im = ax.imshow(sub, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
    ax.set_yticks(range(7)); ax.set_yticklabels(BGEFT, fontsize=8)
    ax.set_xticks(range(len(BASE_IDX))); ax.set_xticklabels([FEATS[i] for i in BASE_IDX], rotation=90, fontsize=6)
    plt.colorbar(im, fraction=0.02, pad=0.01)
    ax.set_title('Tương quan: 7 bge_ft (hàng) × 45 cũ (cột) — |corr| thấp ⇒ tín hiệu mới')
    plt.tight_layout(); plt.show()
    ci = BGEFT.index('bgeft_cosine')
    rels = sorted([(FEATS[BASE_IDX[j]], float(sub[ci, j])) for j in range(len(BASE_IDX))], key=lambda kv: -abs(kv[1]))
    print('bgeft_cosine tương quan mạnh nhất với (top 5 trong 45 cũ):')
    for n, v in rels[:5]: print(f'   {n:24} r={v:+.3f}')
else:
    print('(cần dữ liệu — xem mục 1)')

In [ ]:
if DATA_OK:
    rng = np.random.default_rng(2)
    neg = rng.choice(np.where(yall == 0)[0], size=min(8000, int((yall == 0).sum())), replace=False)
    pos = np.where(yall == 1)[0]
    ic, ib = FEATS.index('bgeft_cosine'), FEATS.index('bge_score')
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(Xall[neg, ib], Xall[neg, ic], s=4, alpha=0.15, c='steelblue', label='non-gold')
    ax.scatter(Xall[pos, ib], Xall[pos, ic], s=10, alpha=0.6, c='crimson', label='gold')
    ax.set_xlabel('bge_score (BGE zero-shot)'); ax.set_ylabel('bgeft_cosine (fine-tuned)')
    ax.set_title('Fine-tuned cosine vs zero-shot BGE — gold tách theo trục bge_ft'); ax.legend()
    plt.tight_layout(); plt.show()
else:
    print('(cần dữ liệu — xem mục 1)')

## 7. L6 thực sự dùng `bge_ft` nhiều cỡ nào? (LightGBM gain importance)

Nạp `results/models/l6_ltr_final_bgeft.features.json` (importances đã học). Nếu chưa có file, dùng số tham chiếu §23.7 trong `FULL_M4_V2_RESULTS.md`. **Trục log** vì `bgeft_cosine` áp đảo (gain ~870k, gấp ~90× đặc trưng kế tiếp).

In [ ]:
impf = ROOT / 'results/models/l6_ltr_final_bgeft.features.json'
if impf.exists():
    imp = json.loads(impf.read_text())['gain_importance']   # [[name, gain], ...] giảm dần
    src = 'đã học (l6_ltr_final_bgeft.features.json)'
else:
    imp = [('bgeft_cosine', 869157), ('bgeft_inv_rank', 9282), ('bgeft_rank', 9273),
           ('bm25_norm', 6624), ('bm25_score', 5979), ('bgeft_margin_top1', 4637),
           ('global_cluster_reliability', 3573), ('bgeft_z', 3444), ('bm25_rank', 2677),
           ('qsc_local_centrality', 2330), ('global_cluster_size', 2106), ('query_length_tokens', 1669)]
    src = 'tham chiếu §23.7 (chưa có file importances cục bộ)'
top = imp[:15][::-1]
names = [n for n, _ in top]; vals = [max(v, 1) for _, v in top]
cols = ['#C44E52' if FEAT2GROUP.get(n) == 'bge_ft' else '#999999' for n in names]
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(names, vals, color=cols); ax.set_xscale('log')
for lbl, n in zip(ax.get_yticklabels(), names):
    if FEAT2GROUP.get(n) == 'bge_ft': lbl.set_fontweight('bold')
ax.set_xlabel('gain (log scale)')
ax.set_title(f'LightGBM gain importance — top 15\nĐỏ = bge_ft | nguồn: {src}')
plt.tight_layout(); plt.show()
n_bge = sum(1 for n, _ in imp[:10] if FEAT2GROUP.get(n) == 'bge_ft')
print(f'bge_ft chiếm {n_bge}/10 vị trí gain cao nhất; bgeft_cosine đứng #1.')

## 8. `bge_ft` kéo gold lên đỉnh (giải thích R@1 48→62)

So sánh **rank của gold** theo từng tín hiệu xếp hạng: `bgeft_rank` (mới) vs `bge_rank`, `m4_rank`, `bm25_rank` (cũ). ECDF càng dốc/sớm càng tốt; bar = % gold lọt top-1/5/10. `bge_ft` đưa nhiều gold về **rank 1** hơn hẳn các tín hiệu cũ.

In [ ]:
if DATA_OK:
    gold = yall == 1
    pal = {'bgeft_rank': 'crimson', 'bge_rank': 'steelblue', 'm4_rank': 'seagreen', 'bm25_rank': 'darkorange'}
    pal = {k: v for k, v in pal.items() if k in FEATS}
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    for name, c in pal.items():
        r = np.sort(np.clip(Xall[gold, FEATS.index(name)], 1, 100))
        ax[0].plot(r, np.linspace(0, 1, len(r)), color=c, label=name, lw=2)
    ax[0].set_xlabel('rank của gold (clip@100)'); ax[0].set_ylabel('tỉ lệ gold ≤ rank')
    ax[0].set_title('ECDF rank-của-gold'); ax[0].legend(); ax[0].grid(alpha=0.3)
    ks = [1, 5, 10]; width = 0.8 / len(pal)
    for j, (name, c) in enumerate(pal.items()):
        r = Xall[gold, FEATS.index(name)]
        ax[1].bar(np.arange(len(ks)) + j*width, [100*np.mean(r <= k) for k in ks], width, color=c, label=name)
    ax[1].set_xticks(np.arange(len(ks)) + width*(len(pal)-1)/2); ax[1].set_xticklabels([f'≤{k}' for k in ks])
    ax[1].set_ylabel('% gold'); ax[1].set_title('Gold lọt top-k theo từng tín hiệu'); ax[1].legend()
    plt.tight_layout(); plt.show()
else:
    print('(cần dữ liệu — xem mục 1)')

## 9. `bge_ft` giúp nhiều nhất ở dataset nào?

Trái: AUC của `bgeft_cosine` theo dataset (khả năng tách gold *trong pool*). Phải: tương quan giữa AUC đó và **Δ R@1 = (L6+bge_ft − L6)** đọc từ §23.2 — dataset nào `bge_ft` tách tốt thì cải thiện cuối lớn (logicbench +40pp, toolqa +16pp, bigcodebench +10pp).

In [ ]:
if DATA_OK:
    from sklearn.metrics import roc_auc_score
    ic = FEATS.index('bgeft_cosine')
    auc_ds = {}
    for ds in DATASETS:
        y = data[ds]['y']; x = data[ds]['X'][:, ic]
        auc_ds[ds] = roc_auc_score(y, x) if len(np.unique(y)) > 1 else np.nan
    # Δ R@1 (L6+bge_ft − L6) từ §23.2 của FULL_M4_V2_RESULTS.md
    r1 = {'theoremqa': (75.84, 75.17), 'logicbench': (67.11, 26.97), 'toolqa': (96.50, 80.42),
          'champ': (30.30, 19.70), 'medcalcbench': (64.09, 60.45), 'bigcodebench': (37.24, 27.34)}
    gain = {ds: r1[ds][0] - r1[ds][1] for ds in DATASETS}
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    ax[0].bar(DATASETS, [auc_ds[ds] for ds in DATASETS], color='crimson')
    ax[0].set_ylim(0.5, 1.0); ax[0].set_title('AUC bgeft_cosine theo dataset (tách gold trong pool)')
    ax[0].tick_params(axis='x', rotation=45); ax[0].grid(axis='y', alpha=0.3)
    ax[1].scatter([auc_ds[ds] for ds in DATASETS], [gain[ds] for ds in DATASETS], s=60, c='crimson')
    for ds in DATASETS:
        ax[1].annotate(ds, (auc_ds[ds], gain[ds]), fontsize=8, xytext=(3, 3), textcoords='offset points')
    ax[1].set_xlabel('AUC bgeft_cosine (local)'); ax[1].set_ylabel('Δ R@1 (L6+bge_ft − L6), pp [§23.2]')
    ax[1].set_title('Tách tốt trong pool ⇒ cải thiện R@1 cuối'); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('(cần dữ liệu — xem mục 1)')

## 10. Ví dụ cụ thể: một query `bge_ft` cứu được

Tìm query mà gold bị `bge_rank` (cũ) xếp thấp (≥20) nhưng `bgeft_rank` (mới) kéo lên **top-1**. In 7 giá trị `bge_ft` của gold + top-5 pool theo `bgeft_cosine` (đánh dấu gold).

In [ ]:
if DATA_OK:
    def query_slices(ds):
        d = data[ds]; off = 0
        for i, iid in enumerate(d['instance_ids']):
            n = int(d['group_sizes'][i]); yield iid, slice(off, off + n); off += n
    ir_old, ir_new = FEATS.index('bge_rank'), FEATS.index('bgeft_rank')
    pick = None
    for ds in DATASETS:
        d = data[ds]
        for iid, sl in query_slices(ds):
            yq = d['y'][sl]; gi = np.where(yq == 1)[0]
            if len(gi) == 0: continue
            g = gi[0]; X = d['X'][sl]
            if X[g, ir_old] >= 20 and X[g, ir_new] <= 1:
                pick = (ds, iid, sl, g); break
        if pick: break
    if pick is None:
        print('(không tìm thấy ví dụ thoả điều kiện — pool quá dễ; nới ngưỡng nếu cần)')
    else:
        ds, iid, sl, g = pick; d = data[ds]; X = d['X'][sl]
        sids = d['skill_ids'][sl.start:sl.stop]; gold_sid = sids[g]
        print(f'dataset={ds}  query={iid}')
        print(f'gold={gold_sid} | bge_rank(cũ)={X[g, ir_old]:.0f} -> bgeft_rank(mới)={X[g, ir_new]:.0f}')
        display(pd.DataFrame({'bge_ft feature': BGEFT, 'giá trị (gold)': np.round(X[g, BGE_IDX], 4)}))
        ic = FEATS.index('bgeft_cosine'); ordr = np.argsort(-X[:, ic])[:5]
        display(pd.DataFrame({'#': range(1, 6), 'skill_id': [sids[i] for i in ordr],
                              'bgeft_cosine': np.round(X[ordr, ic], 4),
                              'is_gold': [sids[i] == gold_sid for i in ordr]}))
else:
    print('(cần dữ liệu — xem mục 1)')

## 11. Trần recall của pool & bước tiếp theo (§24 union)

L6+bge_ft xếp hạng **trong M4 pool** (gold-in-pool ≈97.6%) nên *recall* bị chặn bởi trần này — `bge_ft` cải thiện **thứ tự** (nDCG, R@1) chứ không vượt được recall của bge_ft-alone (full-corpus R@100 ≈99.3%) nếu không nới pool. §23.6: L6+bge_ft vs L6 nDCG@10 **+13.25pp (p≈0)**; vs bge_ft-alone +2.64pp (p≈0).

**§24 — Pool Union** giải quyết trần đó: xếp hạng trên `P_union = P_M4 ∪ TopK_bge_ft` (thêm nhóm 7 cờ `union`: `in_m4_pool, in_bgeft_topk, in_both, missing_pool_features, ...`). Kết quả tốt nhất **L6-union-bgeft@100**: gold-in-pool 97.9%→**100%**, R@100→**99.48**, **macro nDCG@10 = 82.23** (so với L6+bge_ft 80.95, L6-final 67.43).

| Method | R@1 | R@10 | R@100 | nDCG@10 |
|---|---|---|---|---|
| L6-final (45) | 48.34 | 81.07 | 97.10 | 67.43 |
| **L6-final+bge_ft (52)** | **61.85** | **89.73** | 97.21 | **80.95** |
| L6-union-bgeft@100 (59) | 62.89 | 91.63 | **99.48** | **82.23** |
| bge_ft (retriever-only) | 56.99 | 87.87 | 99.26 | 76.85 |
| M7-CE@500 | 45.74 | 84.92 | 96.37 | 69.42 |

_Sinh source union (cho end-task): `python src/kmeans/scripts/ltr/run_l6_union_bgeft.py --K 100` → `results/retrieval/{ds}-l6_union_bgeft_k100.json`. Chi tiết §23–§24 trong `FULL_M4_V2_RESULTS.md`._